# Building a Decision-Based Digital Twin of Rodri

This notebook builds a simple, explainable digital twin of Rodri's passing decisions using StatsBomb Open Data.

The model focuses on four football decision signals:

- field zone
- pressure
- pass direction
- pass progression

The goal is not to predict every action perfectly, but to create a clean analytical workflow that explains how Rodri's decisions change by context.


## 1. Introduction

Rodri is one of the best midfielders in the world at controlling possession, progressing play, and making efficient decisions under pressure.

In this project, we treat his historical event data as the behavioral base for a small decision-based digital twin. The twin learns how often Rodri chooses forward, lateral, or backward passes in different zones and pressure states, then simulates likely decisions in similar situations.


## 2. Dataset

The analysis uses StatsBomb Open Data event files in JSON format.

Only the 24 selected matches where Rodri appears are loaded. This keeps the notebook lighter and avoids loading the full event dataset into memory.

Expected folder structure:

```text
DATA_PATH/
  events/
    16073.json
    16248.json
    ...
```

Update `DATA_PATH` below if your StatsBomb data folder is stored somewhere else.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titleweight": "bold",
})

DATA_PATH = Path(r"C:\Users\giova\Downloads\archive (2)\data")
EVENTS_PATH = DATA_PATH / "events"

PLAYER_NAME = "Rodrigo Hern\u00e1ndez Cascante"

MATCH_IDS_RODRI = [
    "16073", "16248", "266477", "266952", "3788750", "3788762",
    "3794686", "3795108", "3795220", "3825836", "3825873", "3825893",
    "3857255", "3857263", "3857291", "3869220", "3930160", "3930172",
    "3941018", "3942226", "3942752", "3943043", "9717", "9928",
]

ZONE_ORDER = ["Defensive", "Central", "Offensive"]
DIRECTION_ORDER = ["Forward", "Lateral", "Backward"]
PRESSURE_LABELS = {False: "No pressure", True: "Under pressure"}


## 3. Data Loading

The functions below load only the selected event files, add the match id to each event, and fail with useful error messages when files are missing or malformed.

This avoids the main memory problem: the notebook never needs to load every StatsBomb event file, only the match ids used in the project.


In [ ]:
def require_columns(df, columns, df_name="DataFrame"):
    """Raise a clear error if required columns are missing."""
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise KeyError(f"{df_name} is missing required columns: {missing}")


def nested_name(value):
    """Extract the 'name' field from a StatsBomb nested dictionary."""
    if isinstance(value, dict):
        return value.get("name")
    return np.nan


def load_events(match_ids, events_path, keep_columns=None, skip_missing=False):
    """
    Load StatsBomb event JSON files for selected match ids only.

    Parameters
    ----------
    match_ids : list[str]
        Match ids to load.
    events_path : str or pathlib.Path
        Folder containing event JSON files.
    keep_columns : list[str], optional
        Optional subset of columns to keep after loading.
    skip_missing : bool
        If True, missing files are skipped. If False, a missing file raises FileNotFoundError.
    """
    events_path = Path(events_path)
    all_events = []
    missing_files = []

    for match_id in match_ids:
        file_path = events_path / f"{match_id}.json"

        if not file_path.exists():
            missing_files.append(str(file_path))
            if skip_missing:
                continue
            raise FileNotFoundError(f"Missing JSON file for match_id={match_id}: {file_path}")

        try:
            with file_path.open("r", encoding="utf-8") as f:
                match_events = json.load(f)
        except json.JSONDecodeError as exc:
            raise ValueError(f"Invalid JSON file for match_id={match_id}: {file_path}") from exc
        except MemoryError as exc:
            raise MemoryError(
                "MemoryError while loading events. Reduce match_ids or load fewer columns."
            ) from exc

        for event in match_events:
            event["match_id"] = str(match_id)

        all_events.extend(match_events)

    if not all_events:
        raise ValueError("No events were loaded. Check match_ids and events_path.")

    df = pd.DataFrame(all_events)

    if keep_columns is not None:
        existing_columns = [col for col in keep_columns if col in df.columns]
        df = df.loc[:, existing_columns].copy()

    if missing_files and skip_missing:
        print(f"Skipped {len(missing_files)} missing files.")

    return df


def add_basic_event_columns(df):
    """Add player_name and event_type using StatsBomb nested columns."""
    require_columns(df, ["player", "type"], "events")

    clean_df = df.copy()
    clean_df.loc[:, "player_name"] = clean_df["player"].map(nested_name)
    clean_df.loc[:, "event_type"] = clean_df["type"].map(nested_name)
    return clean_df


def filter_player(df, player_name):
    """Return a safe copy containing only events for one player."""
    require_columns(df, ["player_name"], "events")
    player_df = df.loc[df["player_name"].eq(player_name)].copy()

    if player_df.empty:
        available = df["player_name"].dropna().sort_values().unique()[:10]
        raise ValueError(
            f"No events found for '{player_name}'. Example available players: {list(available)}"
        )

    return player_df


In [ ]:
KEEP_COLUMNS = [
    "id", "match_id", "index", "period", "timestamp", "minute", "second",
    "team", "player", "type", "location", "pass", "carry", "shot", "dribble",
    "under_pressure", "possession", "possession_team", "play_pattern",
]

raw_events = load_events(
    match_ids=MATCH_IDS_RODRI,
    events_path=EVENTS_PATH,
    keep_columns=KEEP_COLUMNS,
    skip_missing=False,
)

events = add_basic_event_columns(raw_events)
rodri_events = filter_player(events, PLAYER_NAME)

print(f"Loaded matches: {events['match_id'].nunique()}")
print(f"Total events loaded: {len(events):,}")
print(f"Rodri events: {len(rodri_events):,}")

display(rodri_events[["match_id", "minute", "second", "player_name", "event_type"]].head())


## 4. Data Cleaning

The cleaning step keeps the notebook explicit and reproducible:

- `under_pressure` is converted with `.eq(True)` to avoid `fillna(...).astype(bool)` warnings.
- player and event names are extracted from nested dictionaries.
- filtering always returns `.copy()` to avoid `SettingWithCopyWarning`.
- pass coordinates are extracted only from valid pass events.


In [ ]:
def safe_pressure_flag(series):
    """Convert StatsBomb under_pressure values to clean booleans without fillna warnings."""
    return series.eq(True)


def extract_location_x(location):
    if isinstance(location, list) and len(location) >= 1:
        return location[0]
    return np.nan


def extract_location_y(location):
    if isinstance(location, list) and len(location) >= 2:
        return location[1]
    return np.nan


def extract_pass_end_location(pass_data, index):
    if isinstance(pass_data, dict):
        end_location = pass_data.get("end_location")
        if isinstance(end_location, list) and len(end_location) > index:
            return end_location[index]
    return np.nan


def prepare_passes(df):
    """Prepare valid pass events with start/end coordinates and pressure flag."""
    require_columns(df, ["event_type", "location", "pass"], "events")

    passes = df.loc[df["event_type"].eq("Pass")].copy()

    if "under_pressure" not in passes.columns:
        passes.loc[:, "under_pressure"] = False
    else:
        passes.loc[:, "under_pressure"] = safe_pressure_flag(passes["under_pressure"])

    passes.loc[:, "start_x"] = passes["location"].map(extract_location_x)
    passes.loc[:, "start_y"] = passes["location"].map(extract_location_y)
    passes.loc[:, "end_x"] = passes["pass"].map(lambda value: extract_pass_end_location(value, 0))
    passes.loc[:, "end_y"] = passes["pass"].map(lambda value: extract_pass_end_location(value, 1))

    before = len(passes)
    passes = passes.dropna(subset=["start_x", "start_y", "end_x", "end_y"]).copy()
    removed = before - len(passes)

    if removed > 0:
        print(f"Removed {removed:,} pass events with missing coordinates.")

    if passes.empty:
        raise ValueError("No valid pass events after coordinate extraction.")

    return passes


def assign_field_zone(x):
    """Assign field zone using StatsBomb x coordinate on a 120m pitch."""
    if pd.isna(x):
        return np.nan
    if x < 40:
        return "Defensive"
    if x < 80:
        return "Central"
    return "Offensive"


def classify_pass_direction(row, threshold=5):
    """Classify pass direction from x-coordinate progression."""
    progression = row["end_x"] - row["start_x"]

    if progression > threshold:
        return "Forward"
    if progression < -threshold:
        return "Backward"
    return "Lateral"


def add_pass_features(passes):
    """Add zone, direction, progression, and readable pressure label."""
    require_columns(passes, ["start_x", "end_x", "under_pressure"], "passes")

    featured = passes.copy()
    featured.loc[:, "field_zone"] = featured["start_x"].map(assign_field_zone)
    featured.loc[:, "progression"] = featured["end_x"] - featured["start_x"]
    featured.loc[:, "pass_direction"] = featured.apply(classify_pass_direction, axis=1)
    featured.loc[:, "pressure_label"] = featured["under_pressure"].map(PRESSURE_LABELS)

    return featured

rodri_passes = prepare_passes(rodri_events)
rodri_passes = add_pass_features(rodri_passes)

print(f"Rodri valid passes: {len(rodri_passes):,}")
display(rodri_passes[[
    "match_id", "minute", "start_x", "start_y", "end_x", "end_y",
    "field_zone", "pass_direction", "progression", "pressure_label",
]].head())


## 5. Decision Profile

This section describes Rodri's overall event profile across the selected matches. It gives context before focusing only on passing decisions.


In [ ]:
def add_bar_labels(ax, values, fmt="{:.1f}%", y_offset=0.01):
    """Add readable labels above vertical bars."""
    max_value = max(values) if len(values) else 0
    for patch, value in zip(ax.patches, values):
        ax.text(
            patch.get_x() + patch.get_width() / 2,
            patch.get_height() + max_value * y_offset,
            fmt.format(value),
            ha="center",
            va="bottom",
            fontsize=10,
        )


def plot_percentage_bar(series, title, ylabel="Percentage", color="#2F6B8F"):
    values = series.mul(100)
    ax = values.plot(kind="bar", color=color)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)
    add_bar_labels(ax, values.values)
    plt.tight_layout()
    plt.show()


def plot_value_bar(series, title, ylabel, color="#577590", fmt="{:.2f}"):
    ax = series.plot(kind="bar", color=color)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)
    add_bar_labels(ax, series.values, fmt=fmt, y_offset=0.03)
    plt.tight_layout()
    plt.show()

decision_profile = (
    rodri_events["event_type"]
    .value_counts(normalize=True)
    .loc[lambda s: s.index.isin(["Pass", "Carry", "Shot", "Dribble"])]
    .sort_values(ascending=False)
)

display(decision_profile.to_frame("share"))
plot_percentage_bar(decision_profile, "Rodri Decision Profile by Event Type")


## 6. Spatial Analysis

The next step adds spatial context. Each pass is assigned to one of three zones using the starting x-coordinate:

- Defensive: x < 40
- Central: 40 <= x < 80
- Offensive: x >= 80

This keeps the model simple and easy to explain for a portfolio project.


In [ ]:
passes_by_zone = (
    rodri_passes["field_zone"]
    .value_counts(normalize=True)
    .reindex(ZONE_ORDER)
    .dropna()
)

display(passes_by_zone.to_frame("share"))
plot_percentage_bar(passes_by_zone, "Rodri Pass Distribution by Field Zone", color="#4D908E")


## 7. Passing Direction

Pass direction is based on real x-coordinate movement:

- Forward: end_x - start_x > 5
- Backward: end_x - start_x < -5
- Lateral: movement between -5 and +5

A small threshold prevents tiny coordinate changes from being over-interpreted.


In [ ]:
pass_direction_share = (
    rodri_passes["pass_direction"]
    .value_counts(normalize=True)
    .reindex(DIRECTION_ORDER)
    .dropna()
)

display(pass_direction_share.to_frame("share"))
plot_percentage_bar(pass_direction_share, "Rodri Pass Direction Distribution", color="#43AA8B")


In [ ]:
direction_by_zone = pd.crosstab(
    rodri_passes["field_zone"],
    rodri_passes["pass_direction"],
    normalize="index",
).reindex(index=ZONE_ORDER, columns=DIRECTION_ORDER)

display(direction_by_zone)

ax = direction_by_zone.mul(100).plot(
    kind="bar",
    stacked=True,
    color=["#43AA8B", "#6C757D", "#B56576"],
)
ax.set_title("Rodri Pass Direction by Field Zone")
ax.set_ylabel("Percentage")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Direction", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 8. Progression Analysis

Progression measures how many meters the pass moves the ball forward on the x-axis.

Positive values mean the pass advances play. Negative values mean the pass moves the ball backward.


In [ ]:
progression_summary = rodri_passes["progression"].agg(["mean", "median", "std", "count"])
progression_by_zone = (
    rodri_passes
    .groupby("field_zone")["progression"]
    .agg(["mean", "median", "count"])
    .reindex(ZONE_ORDER)
)

display(progression_summary.to_frame("progression"))
display(progression_by_zone)
plot_value_bar(
    progression_by_zone["mean"],
    "Average Pass Progression by Field Zone",
    ylabel="Meters gained",
    color="#F8961E",
)


## 9. Pressure Analysis

Pressure is a key part of the digital twin because the same player can behave differently when pressed.

Here we compare Rodri's progression with and without pressure, especially in the defensive zone where pressure can strongly shape decision-making.


In [ ]:
pressure_summary = (
    rodri_passes
    .groupby("pressure_label")["progression"]
    .agg(["mean", "median", "count"])
    .reindex(["No pressure", "Under pressure"])
)

display(pressure_summary)
plot_value_bar(
    pressure_summary["mean"],
    "Average Pass Progression by Pressure State",
    ylabel="Meters gained",
    color="#F3722C",
)


In [ ]:
defensive_pressure = (
    rodri_passes
    .loc[rodri_passes["field_zone"].eq("Defensive")]
    .groupby("pressure_label")["progression"]
    .agg(["mean", "median", "count"])
    .reindex(["No pressure", "Under pressure"])
)

display(defensive_pressure)
plot_value_bar(
    defensive_pressure["mean"],
    "Rodri Defensive-Zone Progression by Pressure State",
    ylabel="Meters gained",
    color="#F94144",
)


## 10. Digital Twin Simulation

The digital twin is an explainable probabilistic model.

For each combination of field zone and pressure state, it estimates:

- probability of each pass direction
- average progression for each pass direction
- sample size supporting the estimate

When a requested combination is not present, the simulation falls back to broader data instead of failing silently.


In [ ]:
def create_decision_model(passes):
    """Create an explainable probabilistic decision model by zone, pressure, and direction."""
    require_columns(
        passes,
        ["field_zone", "under_pressure", "pass_direction", "progression"],
        "passes",
    )

    grouped = (
        passes
        .groupby(["field_zone", "under_pressure", "pass_direction"], dropna=False)
        .agg(
            count=("pass_direction", "size"),
            avg_progression=("progression", "mean"),
            std_progression=("progression", "std"),
        )
        .reset_index()
    )

    total_by_context = grouped.groupby(["field_zone", "under_pressure"])["count"].transform("sum")
    grouped.loc[:, "probability"] = grouped["count"] / total_by_context
    grouped.loc[:, "pressure_label"] = grouped["under_pressure"].map(PRESSURE_LABELS)

    return grouped.sort_values(["field_zone", "under_pressure", "pass_direction"]).reset_index(drop=True)


def get_context_distribution(model, zone, under_pressure):
    """Return the model rows for one context, with fallbacks for missing combinations."""
    require_columns(
        model,
        ["field_zone", "under_pressure", "pass_direction", "probability"],
        "model",
    )

    under_pressure = bool(under_pressure)
    context = model.loc[
        model["field_zone"].eq(zone) & model["under_pressure"].eq(under_pressure)
    ].copy()

    fallback_used = "exact context"

    if context.empty:
        context = model.loc[model["field_zone"].eq(zone)].copy()
        fallback_used = "zone-only fallback"

    if context.empty:
        context = model.copy()
        fallback_used = "global fallback"

    if context.empty:
        raise ValueError("The decision model is empty. Build it from valid pass data first.")

    # Re-normalize probabilities after fallback.
    context = (
        context
        .groupby("pass_direction", as_index=False)
        .agg(
            count=("count", "sum"),
            avg_progression=("avg_progression", "mean"),
            std_progression=("std_progression", "mean"),
        )
    )
    context.loc[:, "probability"] = context["count"] / context["count"].sum()
    context.loc[:, "fallback_used"] = fallback_used

    return context


def simulate_player_behavior(model, zone, under_pressure, n=10, random_state=42):
    """Simulate Rodri-like pass decisions for a zone and pressure state."""
    if zone not in ZONE_ORDER:
        raise ValueError(f"Unknown zone '{zone}'. Use one of: {ZONE_ORDER}")

    rng = np.random.default_rng(random_state)
    distribution = get_context_distribution(model, zone, under_pressure)

    directions = distribution["pass_direction"].to_numpy()
    probabilities = distribution["probability"].to_numpy()
    choices = rng.choice(directions, size=n, p=probabilities)

    lookup = distribution.set_index("pass_direction")
    rows = []

    for choice in choices:
        avg = lookup.loc[choice, "avg_progression"]
        std = lookup.loc[choice, "std_progression"]
        std = 0 if pd.isna(std) else std
        simulated_progression = rng.normal(avg, std) if std > 0 else avg

        rows.append({
            "zone": zone,
            "under_pressure": bool(under_pressure),
            "pressure_label": PRESSURE_LABELS[bool(under_pressure)],
            "simulated_direction": choice,
            "expected_progression": avg,
            "simulated_progression": simulated_progression,
            "fallback_used": distribution["fallback_used"].iloc[0],
        })

    return pd.DataFrame(rows)

model = create_decision_model(rodri_passes)
display(model.head(12))


In [ ]:
simulation = simulate_player_behavior(
    model=model,
    zone="Defensive",
    under_pressure=True,
    n=20,
    random_state=7,
)

display(simulation)

simulation_summary = simulation["simulated_direction"].value_counts(normalize=True).reindex(DIRECTION_ORDER).dropna()
plot_percentage_bar(
    simulation_summary,
    "Simulated Rodri Decisions: Defensive Zone Under Pressure",
    color="#277DA1",
)


## 11. Key Insights

Main findings from the selected 24-match sample:

- Rodri's passing profile is not static: it changes by field zone and pressure state.
- Forward passing is strongest in deeper zones, where progression has more available space.
- In advanced zones, average progression naturally decreases because there is less pitch left to attack.
- Defensive-zone passes under pressure can show higher average progression, which suggests that pressure situations often force or invite more vertical escape passes.
- The digital twin is useful because it links decision probabilities to context instead of only describing global averages.


## 12. Limitations

This is a portfolio-level analytical model, not a full player simulation engine.

Current limitations:

- the model uses only event data, not tracking data
- pressure is a binary StatsBomb event flag
- pass difficulty, body orientation, opponent location, receiver quality, and tactical phase are not modeled
- the simulation estimates decision style, not pass success or tactical optimality
- sample size can be small for specific zone-pressure combinations


## 13. Future Improvements

Possible next steps:

- include pass outcome and completion probability
- separate open play, set pieces, and transition moments
- compare Rodri with similar midfielders
- add pass length and pass angle
- model receiver zones and next action value
- create a small dashboard version for GitHub or LinkedIn storytelling
- add unit tests if these functions are moved into a separate `.py` module


In [ ]:
print("Notebook completed successfully.")
print(f"Rodri events: {len(rodri_events):,}")
print(f"Rodri valid passes: {len(rodri_passes):,}")
print(f"Average pass progression: {rodri_passes['progression'].mean():.2f} meters")
